In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from dall_e import map_pixels, unmap_pixels, load_model
from dall_e import Encoder, Decoder
import torch

import onnxruntime as ort
import numpy as np

In [3]:
device = torch.device("cuda")
enc: Encoder = load_model("https://cdn.openai.com/dall-e/encoder.pkl", device)
dec: Decoder = load_model("https://cdn.openai.com/dall-e/decoder.pkl", device)

In [4]:
enc_cpu = enc.cpu().eval()

dummy_img = torch.randn(1, 3, 480, 640)
torch.onnx.export(
    enc_cpu,
    dummy_img,
    "./enc.onnx",
    export_params=True,
    input_names=["pixels"],
    dynamic_axes={"pixels": {0: "batch", 2: "height", 3: "width"}},
)
print("ONNX model saved: enc.onnx")

/tmp/ipykernel_69452/1936092093.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `Encoder([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Encoder([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX model saved: enc.onnx


In [ ]:
# trtexec --onnx=enc.onnx --saveEngine=enc.trt --fp16

In [ ]:
trtexec \
  --onnx=./enc.onnx \
  --saveEngine=./enc.trt \
  --fp16 \
  --workspace=4096 \
  --useSpinWait

In [ ]:
[05/03/2026-17:25:35] [I] Throughput: 5200.7 qps
[05/03/2026-17:25:35] [I] Latency: min = 0.195068 ms, max = 0.385681 ms, mean = 0.204975 ms, median = 0.204468 ms, percentile(90%) = 0.209106 ms, percentile(95%) = 0.211182 ms, percentile(99%) = 0.216583 ms
[05/03/2026-17:25:35] [I] Enqueue Time: min = 0.106934 ms, max = 0.43927 ms, mean = 0.124905 ms, median = 0.127197 ms, percentile(90%) = 0.135742 ms, percentile(95%) = 0.138611 ms, percentile(99%) = 0.153076 ms
[05/03/2026-17:25:35] [I] H2D Latency: min = 0.00476074 ms, max = 0.0446777 ms, mean = 0.00983314 ms, median = 0.0090332 ms, percentile(90%) = 0.0134277 ms, percentile(95%) = 0.0147705 ms, percentile(99%) = 0.0180664 ms
[05/03/2026-17:25:35] [I] GPU Compute Time: min = 0.182617 ms, max = 0.368713 ms, mean = 0.187595 ms, median = 0.1875 ms, percentile(90%) = 0.189514 ms, percentile(95%) = 0.190308 ms, percentile(99%) = 0.192383 ms
[05/03/2026-17:25:35] [I] D2H Latency: min = 0.00488281 ms, max = 0.0107422 ms, mean = 0.00754818 ms, median = 0.00756836 ms, percentile(90%) = 0.00897217 ms, percentile(95%) = 0.00927734 ms, percentile(99%) = 0.00976562 ms
[05/03/2026-17:25:35] [I] Total Host Walltime: 3.00056 s
[05/03/2026-17:25:35] [I] Total GPU Compute Time: 2.92741 s

In [ ]:
[05/03/2026-17:30:40] [I] === Performance summary ===
[05/03/2026-17:30:40] [I] Throughput: 5432.36 qps
[05/03/2026-17:30:40] [I] Latency: min = 0.187744 ms, max = 0.243408 ms, mean = 0.193056 ms, median = 0.192871 ms, percentile(90%) = 0.19458 ms, percentile(95%) = 0.19516 ms, percentile(99%) = 0.198303 ms
[05/03/2026-17:30:40] [I] Enqueue Time: min = 0.107178 ms, max = 0.207214 ms, mean = 0.118099 ms, median = 0.117676 ms, percentile(90%) = 0.123657 ms, percentile(95%) = 0.127563 ms, percentile(99%) = 0.139526 ms
[05/03/2026-17:30:40] [I] H2D Latency: min = 0.00488281 ms, max = 0.0474854 ms, mean = 0.00742402 ms, median = 0.00732422 ms, percentile(90%) = 0.00775146 ms, percentile(95%) = 0.00793457 ms, percentile(99%) = 0.0125732 ms
[05/03/2026-17:30:40] [I] GPU Compute Time: min = 0.178223 ms, max = 0.233154 ms, mean = 0.181983 ms, median = 0.181885 ms, percentile(90%) = 0.18335 ms, percentile(95%) = 0.183838 ms, percentile(99%) = 0.184631 ms
[05/03/2026-17:30:40] [I] D2H Latency: min = 0.00268555 ms, max = 0.00537109 ms, mean = 0.0036503 ms, median = 0.00366211 ms, percentile(90%) = 0.00427246 ms, percentile(95%) = 0.00439453 ms, percentile(99%) = 0.00476074 ms
[05/03/2026-17:30:40] [I] Total Host Walltime: 3.00054 s
[05/03/2026-17:30:40] [I] Total GPU Compute Time: 2.96632 s